# 🎬 ClipDistrict Studio on Google Colab (free)

Runs the whole app in Google's cloud: **nothing is installed on your PC**.

1. Menu **Runtime → Change runtime type → T4 GPU** → Save (faster subtitles; CPU also works).
2. Menu **Runtime → Run all**.
3. Click **👉 Open ClipDistrict Studio** under step 2.
4. When you're done, run step 4 to download your Shorts. Everything else is deleted when the session ends.

Free Colab sessions stop after some time or when the tab is idle, so keep this tab open while you clip.


In [ ]:
#@title 1 · Install (about 2 minutes)
WHISPER_MODEL = "small" #@param ["base", "small", "medium"]
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
#@markdown Only if your repo is private: add a Colab secret (🔑 on the left) named `GITHUB_TOKEN`.

import os, subprocess
repo = "github.com/eyasahbeni/clipdistrict-studio"
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
    print("🔑 GITHUB_TOKEN found")
except Exception as err:
    print(f"🔑 No GITHUB_TOKEN ({type(err).__name__}). Fine if the repo is public.")
url = f"https://x-access-token:{token}@{repo}" if token else f"https://{repo}"
APP = "/content/clipdistrict-studio"
if os.path.isdir(APP):
    subprocess.run(["git", "-C", APP, "pull", "-q"], check=False)
else:
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, APP], capture_output=True, text=True)
    if r.returncode:
        print((r.stderr or "").replace(token, "***") if token else r.stderr)
        raise SystemExit("❌ Couldn't download the app. Private repo? Check the GITHUB_TOKEN secret "
                         "(Contents: Read-only on clipdistrict-studio) and that notebook access is ON.")
    print("📦 App downloaded")

subprocess.run("apt-get -qq install -y ffmpeg > /dev/null 2>&1", shell=True)
r = subprocess.run(["pip", "-q", "install", "-r", f"{APP}/requirements.txt"], capture_output=True, text=True)
if r.returncode:
    print(r.stderr[-2000:])
    raise SystemExit("Install failed, see the error above")

HOME_DIR = "/content/workspace"
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    HOME_DIR = "/content/drive/MyDrive/ClipDistrict"
os.makedirs(HOME_DIR, exist_ok=True)

import shutil
gpu = shutil.which("nvidia-smi") is not None
print("✅ Installed.", "GPU on 🚀" if gpu else "No GPU (works, subtitles are slower)")


In [ ]:
#@title 2 · Start the app
import os, subprocess, time, urllib.request
if "proc" in globals() and proc.poll() is None:
    proc.terminate(); time.sleep(1)
env = dict(os.environ, WHISPER_MODEL=WHISPER_MODEL, CLIPDISTRICT_HOME=HOME_DIR, PYTHONUNBUFFERED="1")
log = open("/content/clipdistrict.log", "w")
proc = subprocess.Popen(["python", "-m", "clipdistrict", "app", "--host", "127.0.0.1", "--port", "8765", "--no-browser"],
                        cwd=APP, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8765/api/status", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/clipdistrict.log").read()[-3000:])
    raise SystemExit("The app didn't start, see the log above")

from google.colab import output
print("✅ Running. Click the link below (it opens in a new tab). Keep this Colab tab open.")
output.serve_kernel_port_as_window(8765, anchor_text="👉 Open ClipDistrict Studio")


In [ ]:
#@title 3 · Show the app log (run this if something looks stuck)
print(open("/content/clipdistrict.log").read()[-4000:])


In [ ]:
#@title 4 · Download all your Shorts as one ZIP
import glob, os, zipfile
from google.colab import files
clips = glob.glob(f"{HOME_DIR}/projects/*/clips/*.mp4")
if not clips:
    print("No Shorts yet. Render some in the app first.")
else:
    out = "/content/clipdistrict-shorts.zip"
    with zipfile.ZipFile(out, "w") as z:
        for i, path in enumerate(sorted(clips, key=os.path.getmtime), 1):
            z.write(path, f"short-{i:02d}.mp4")
    print(f"{len(clips)} Shorts")
    files.download(out)
